# STAIR-v3.1: ClipFuse-Consensus — Prior-Preserving Fusion + Cross-Modal Consensus Boosting

**Key Innovations:**
- **Prior-Preserving Weighting:** Preserves domain structural prior (Text k=5, Visual k=1 → 83.3% Text : 16.7% Visual).
- **Cross-Modal Consensus Boosting (α=0.5):** Overlapping edges in BOTH Text and Visual kNN receive ×1.5 weight multiplier.

| Parameter | Value |
|---|---|
| Dataset | Amazon2014Baby + Amazon2014Sports |
| Epochs | 500 |
| Neighbors | 5-1 (83.3% Text : 16.7% Visual) |
| Consensus Boost (α) | 0.5 (×1.5 multiplier for overlapping edges) |
| δ (floor) | 0.3 |

In [ ]:
# Cell 1: Setup & Install
import os, shutil

os.chdir('/kaggle/working')

repo = 'STAIR-Enhanced'
if os.path.exists(repo):
    shutil.rmtree(repo)
os.system('git clone https://github.com/ThanhChuong12/STAIR-Enhanced.git')

os.system('pip install nvidia-ml-py -q')
os.system('pip install torchdata==0.6.1 --no-deps -q')
os.system('pip install freerec==0.9.7 -q')
os.system('pip install torch_geometric -q')
os.system('pip install prettytable -q')

import torch, platform, freerec
print('Python  :', platform.python_version())
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
print('freerec :', freerec.__version__)
print('Environment ready')


In [ ]:
# Cell 2: Copy dataset files
import os, shutil

DATA_ROOT = '/kaggle/data'
os.makedirs(DATA_ROOT, exist_ok=True)

def copy_dataset(keywords, full_name):
    dest = os.path.join(DATA_ROOT, full_name)
    os.makedirs(dest, exist_ok=True)
    copied = []
    for root, _, files in os.walk('/kaggle/input'):
        if any(kw.lower() in root.lower() for kw in keywords):
            for f in files:
                if f.endswith(('.npy', '.pkl', '.txt', '.inter', '.item')):
                    shutil.copy(os.path.join(root, f), os.path.join(dest, f))
                    copied.append(f)
    print(f'[{full_name}] {len(copied)} files copied')

copy_dataset(['baby', 'amazon2014baby'],     'Amazon2014Baby_550_MMRec')
copy_dataset(['sports', 'amazon2014sports'], 'Amazon2014Sports_550_MMRec')
print('Data ready at', DATA_ROOT)


In [ ]:
# Cell 3: ClipFuse-v3.1 Structural Diagnostics
import os, sys, pickle, warnings, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, '/kaggle/working/STAIR-Enhanced')
warnings.filterwarnings('ignore')

DATA_ROOT = '/kaggle/data'
DATASETS  = ['Amazon2014Baby_550_MMRec', 'Amazon2014Sports_550_MMRec']
MFILES    = ['textual_modality.pkl', 'visual_modality.pkl']

def load_feat(ds, mf):
    with open(os.path.join(DATA_ROOT, ds, mf), 'rb') as f:
        feat = pickle.load(f)
    if not isinstance(feat, torch.Tensor):
        feat = torch.tensor(feat, dtype=torch.float32)
    return feat.float()

def build_knn_sims(feat, k):
    feat_n = F.normalize(feat, p=2, dim=-1)
    sim = feat_n @ feat_n.t()
    sim.fill_diagonal_(-10.)
    topk_vals, _ = sim.topk(k=k, dim=-1)
    return topk_vals.clamp(min=0.).mean().item()

K_TEXT, K_VIS = 5, 1
ALPHA_CONSENSUS = 0.5

for ds in DATASETS:
    ds_name = 'Baby' if 'Baby' in ds else 'Sports'
    try:
        ft = load_feat(ds, MFILES[0])
        fv = load_feat(ds, MFILES[1])
        mean_sim_t = build_knn_sims(ft, K_TEXT)
        mean_sim_v = build_knn_sims(fv, K_VIS)
        disc_t = max(0., 1. - mean_sim_t)
        disc_v = max(0., 1. - mean_sim_v)
        exp_v = math.exp(disc_v); exp_t = math.exp(disc_t)
        c_v = max(0.3, min(0.7, exp_v / (exp_v + exp_t)))
        c_t = 1.0 - c_v
        w_t = K_TEXT * c_t; w_v = K_VIS * c_v
        pct_t = w_t / (w_t + w_v) * 100; pct_v = 100 - pct_t
        print(f'[{ds_name}] mean_sim: t={mean_sim_t:.4f}, v={mean_sim_v:.4f}')
        print(f'  confidence: c_t={c_t:.4f}, c_v={c_v:.4f}')
        print(f'  weight pool: Text={w_t:.3f} ({pct_t:.1f}%), Visual={w_v:.3f} ({pct_v:.1f}%)')
    except FileNotFoundError:
        print(f'[WARN] Data not found for {ds_name}. Run Cell 2 first.')

print('Diagnostics done.')


In [ ]:
# Cell 4: Train STAIR-v3.1 on Baby
import os, subprocess, time, shutil

os.chdir('/kaggle/working/STAIR-Enhanced')

log_path_baby = '/kaggle/working/log_stair_v3_baby.txt'

print('Training STAIR-v3.1 (ClipFuse-Consensus) on Baby...')
t0 = time.time()
result = subprocess.run(
    ['python', 'main_v3.py',
     '--root', '/kaggle/data',
     '--dataset', 'Amazon2014Baby_550_MMRec',
     '--epochs', '500', '--batch-size', '1024',
     '--embedding-dim', '64', '--num-layers', '3',
     '--num-neighbors', '5-1', '--conf-delta', '0.3',
     '--conf-temp', '1.0', '--alpha-consensus', '0.5',
     '--optimizer', 'adamwsevo', '--lr', '1e-3',
     '--weight-decay', '0.1', '--seed', '1'],
    stdout=open(log_path_baby, 'w'),
    stderr=subprocess.STDOUT, text=True
)
elapsed = (time.time() - t0) / 60
print(f'Done in {elapsed:.1f} min | rc={result.returncode}')

with open(log_path_baby) as f:
    lines = f.readlines()
if result.returncode != 0:
    print('\n[ERROR] Last 30 lines:')
    print(''.join(lines[-30:]))
else:
    print('\nLast 5 lines:')
    print(''.join(lines[-5:]))

os.makedirs('/kaggle/working/STAIR-Enhanced/logs', exist_ok=True)
shutil.copy(log_path_baby, '/kaggle/working/STAIR-Enhanced/logs/log_stair_v3_baby.txt')


In [ ]:
# Cell 5: Train STAIR-v3.1 on Sports
import os, subprocess, time, shutil

os.chdir('/kaggle/working/STAIR-Enhanced')

log_path_sports = '/kaggle/working/log_stair_v3_sports.txt'

print('Training STAIR-v3.1 (ClipFuse-Consensus) on Sports...')
t0 = time.time()
result = subprocess.run(
    ['python', 'main_v3.py',
     '--root', '/kaggle/data',
     '--dataset', 'Amazon2014Sports_550_MMRec',
     '--epochs', '500', '--batch-size', '1024',
     '--embedding-dim', '64', '--num-layers', '3',
     '--num-neighbors', '5-1', '--conf-delta', '0.3',
     '--conf-temp', '1.0', '--alpha-consensus', '0.5',
     '--optimizer', 'adamwsevo', '--lr', '1e-3',
     '--weight-decay', '0.1', '--seed', '1'],
    stdout=open(log_path_sports, 'w'),
    stderr=subprocess.STDOUT, text=True
)
elapsed = (time.time() - t0) / 60
print(f'Done in {elapsed:.1f} min | rc={result.returncode}')

with open(log_path_sports) as f:
    lines = f.readlines()
if result.returncode != 0:
    print('\n[ERROR] Last 30 lines:')
    print(''.join(lines[-30:]))
else:
    print('\nLast 5 lines:')
    print(''.join(lines[-5:]))

os.makedirs('/kaggle/working/STAIR-Enhanced/logs', exist_ok=True)
shutil.copy(log_path_sports, '/kaggle/working/STAIR-Enhanced/logs/log_stair_v3_sports.txt')


In [ ]:
# Cell 6: HARDCODED GROUND-TRUTH DATA — Extracted directly from log files
# Source: log_stair_v3_baby.txt, log_stair_v3_sports.txt
# NO PARSING. NO REGEX. 100% reliable.

# ── BABY: Per-epoch validation metrics (101 checkpoints at eval_step=5) ──────
baby_val_epochs = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90,
 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 145, 150, 155, 160, 165, 170,
 175, 180, 185, 190, 195, 200, 205, 210, 215, 220, 225, 230, 235, 240, 245, 250,
 255, 260, 265, 270, 275, 280, 285, 290, 295, 300, 305, 310, 315, 320, 325, 330,
 335, 340, 345, 350, 355, 360, 365, 370, 375, 380, 385, 390, 395, 400, 405, 410,
 415, 420, 425, 430, 435, 440, 445, 450, 455, 460, 465, 470, 475, 480, 485, 490, 495, 500]

baby_r10 = [0.024, 0.0486, 0.0547, 0.057, 0.0579, 0.0592, 0.059, 0.0589, 0.0589, 0.059,
 0.0596, 0.0599, 0.06, 0.0596, 0.0598, 0.0605, 0.0601, 0.0604, 0.0601, 0.0602,
 0.0601, 0.0601, 0.0601, 0.0604, 0.0609, 0.0607, 0.0608, 0.0612, 0.0609, 0.0608,
 0.0607, 0.0608, 0.0611, 0.0612, 0.0608, 0.0609, 0.0613, 0.061, 0.0609, 0.0607,
 0.0603, 0.0602, 0.0603, 0.0605, 0.0604, 0.0602, 0.0603, 0.0604, 0.0607, 0.0606,
 0.0607, 0.0607, 0.0616, 0.0607, 0.0608, 0.0606, 0.0606, 0.0608, 0.0607, 0.0605,
 0.0604, 0.0605, 0.0602, 0.0606, 0.0605, 0.0604, 0.0605, 0.0605, 0.0605, 0.0603,
 0.0604, 0.0605, 0.0604, 0.0604, 0.0601, 0.0602, 0.0603, 0.0602, 0.0601, 0.06,
 0.06, 0.0602, 0.06, 0.0598, 0.0596, 0.0598, 0.0598, 0.0595, 0.0593, 0.0594,
 0.059, 0.059, 0.0588, 0.059, 0.059, 0.0587, 0.0589, 0.059, 0.0588, 0.059, 0.0591]

baby_r20 = [0.0361, 0.0727, 0.0851, 0.0882, 0.0894, 0.0908, 0.0924, 0.0927, 0.0925, 0.0932,
 0.0944, 0.0941, 0.0937, 0.0936, 0.0945, 0.0938, 0.0933, 0.0925, 0.0927, 0.0929,
 0.0928, 0.093, 0.0937, 0.0929, 0.093, 0.0928, 0.0932, 0.0937, 0.0935, 0.0935,
 0.0936, 0.0935, 0.0938, 0.0938, 0.0935, 0.0937, 0.0942, 0.0938, 0.0932, 0.0932,
 0.0928, 0.0936, 0.093, 0.0932, 0.0928, 0.0926, 0.0929, 0.0927, 0.0933, 0.0929,
 0.0931, 0.0933, 0.0941, 0.0928, 0.0929, 0.0926, 0.0927, 0.0929, 0.0931, 0.0928,
 0.0924, 0.0924, 0.0923, 0.0927, 0.0927, 0.0924, 0.0926, 0.0923, 0.0921, 0.092,
 0.0921, 0.0923, 0.0921, 0.0919, 0.0918, 0.0919, 0.0918, 0.0917, 0.0917, 0.0916,
 0.0915, 0.0916, 0.0912, 0.0912, 0.0908, 0.0912, 0.0909, 0.0906, 0.0905, 0.0905,
 0.0901, 0.0901, 0.09, 0.0901, 0.09, 0.0896, 0.0898, 0.0898, 0.0895, 0.0898, 0.0901]

baby_n10 = [0.0129, 0.0263, 0.029, 0.0301, 0.0305, 0.031, 0.0311, 0.0311, 0.0311, 0.0312,
 0.0315, 0.0316, 0.0318, 0.0316, 0.0316, 0.032, 0.0319, 0.032, 0.0318, 0.0319,
 0.0319, 0.032, 0.0323, 0.0321, 0.0321, 0.0322, 0.0323, 0.0325, 0.0324, 0.0323,
 0.0323, 0.0324, 0.0326, 0.0326, 0.0324, 0.0324, 0.0328, 0.0326, 0.0325, 0.0324,
 0.0322, 0.0322, 0.0322, 0.0323, 0.0322, 0.032, 0.0321, 0.0322, 0.0324, 0.0322,
 0.0323, 0.0324, 0.0328, 0.0322, 0.0323, 0.0322, 0.0322, 0.0323, 0.0323, 0.0322,
 0.032, 0.0321, 0.032, 0.0321, 0.0322, 0.0321, 0.0322, 0.0321, 0.032, 0.032,
 0.032, 0.032, 0.032, 0.032, 0.0319, 0.0319, 0.0319, 0.0319, 0.0319, 0.0318,
 0.0317, 0.0318, 0.0318, 0.0316, 0.0315, 0.0317, 0.0316, 0.0315, 0.0314, 0.0314,
 0.0313, 0.0313, 0.0312, 0.0313, 0.0313, 0.031, 0.0312, 0.0312, 0.0311, 0.0312, 0.0313]

baby_n20 = [0.016, 0.0324, 0.0367, 0.038, 0.0385, 0.0391, 0.0396, 0.0397, 0.0397, 0.0399,
 0.0403, 0.0403, 0.0404, 0.0403, 0.0405, 0.0404, 0.0403, 0.0402, 0.0404, 0.0406,
 0.0406, 0.0406, 0.041, 0.0409, 0.0408, 0.0407, 0.0409, 0.0411, 0.0408, 0.041,
 0.041, 0.0411, 0.0411, 0.0409, 0.0409, 0.0406, 0.0407, 0.0411, 0.0407, 0.0411,
 0.0407, 0.0411, 0.0411, 0.0407, 0.0406, 0.0407, 0.0408, 0.0406, 0.0408, 0.0408,
 0.0408, 0.0406, 0.0409, 0.0405, 0.0406, 0.0403, 0.0407, 0.041, 0.0406, 0.0408,
 0.0409, 0.0408, 0.0408, 0.0408, 0.0407, 0.0408, 0.0407, 0.0407, 0.0406, 0.0405,
 0.0405, 0.0404, 0.0406, 0.0405, 0.0405, 0.0403, 0.0404, 0.0405, 0.0404, 0.0404,
 0.0403, 0.0403, 0.0406, 0.0403, 0.0403, 0.0402, 0.0402, 0.0403, 0.0404, 0.0402,
 0.0401, 0.0399, 0.04, 0.0397, 0.04, 0.0399, 0.0399, 0.0396, 0.04, 0.0399, 0.0396]

# ── BABY: Per-epoch training loss ──────────────────────────────────────────
baby_loss_epochs = list(range(1, 501))
baby_loss_vals = [
 0.63021, 0.61224, 0.5902, 0.56276, 0.53023, 0.4949, 0.45847, 0.42394, 0.39187, 0.36279,
 0.33778, 0.31463, 0.29421, 0.27694, 0.26097, 0.24676, 0.23364, 0.22266, 0.21157, 0.20161,
 0.19274, 0.18458, 0.17807, 0.17072, 0.16278, 0.15767, 0.15147, 0.1457, 0.14129, 0.13681,
 0.1316, 0.12743, 0.12366, 0.12034, 0.11703, 0.11327, 0.10904, 0.10709, 0.10393, 0.10153,
 0.09837, 0.09608, 0.09393, 0.0916, 0.089, 0.08684, 0.08576, 0.08337, 0.08131, 0.07999,
 0.07821, 0.07697, 0.07534, 0.07317, 0.07203, 0.07088, 0.06984, 0.06912, 0.0674, 0.06647,
 0.06519, 0.06373, 0.06283, 0.06205, 0.0614, 0.06007, 0.05919, 0.05818, 0.05748, 0.05693,
 0.05551, 0.05553, 0.05445, 0.05392, 0.0535, 0.05258, 0.05173, 0.05166, 0.05051, 0.05016,
 0.04982, 0.04923, 0.04831, 0.04786, 0.04755, 0.04664, 0.04652, 0.04554, 0.04573, 0.0454,
 0.04452, 0.04431, 0.04376, 0.04369, 0.04295, 0.04276, 0.04224, 0.04146, 0.0414, 0.0414,
 0.04064, 0.04056, 0.04035, 0.03978, 0.03991, 0.03953, 0.03933, 0.03883, 0.03874, 0.03815,
 0.03819, 0.03746, 0.03742, 0.03716, 0.03727, 0.03667, 0.03655, 0.03627, 0.03605, 0.03594,
 0.03599, 0.03566, 0.03529, 0.03495, 0.03456, 0.03471, 0.0348, 0.03444, 0.03419, 0.03398,
 0.0339, 0.03365, 0.03361, 0.03316, 0.03306, 0.03299, 0.03308, 0.03268, 0.03216, 0.03213,
 0.03271, 0.03213, 0.03145, 0.032, 0.03182, 0.0314, 0.03181, 0.03109, 0.03104, 0.0311,
 0.03096, 0.03078, 0.03045, 0.03062, 0.0301, 0.03023, 0.03014, 0.03031, 0.02996, 0.02995,
 0.02965, 0.02966, 0.02972, 0.02946, 0.02942, 0.02927, 0.02882, 0.02888, 0.02902, 0.02851,
 0.02901, 0.02875, 0.02846, 0.02842, 0.02845, 0.02814, 0.02803, 0.0281, 0.02809, 0.02799,
 0.02809, 0.02783, 0.02783, 0.02761, 0.02754, 0.02767, 0.02731, 0.0276, 0.02748, 0.02722,
 0.02736, 0.02701, 0.02693, 0.02688, 0.02681, 0.02688, 0.02687, 0.02689, 0.02667, 0.02657,
 0.02642, 0.02654, 0.02632, 0.02631, 0.02609, 0.02609, 0.02572, 0.02614, 0.02604, 0.02583,
 0.02591, 0.02611, 0.02586, 0.02581, 0.02554, 0.02566, 0.02534, 0.02562, 0.02568, 0.0252,
 0.02544, 0.02533, 0.02516, 0.02511, 0.02503, 0.02529, 0.02504, 0.02483, 0.02501, 0.02447,
 0.02494, 0.0251, 0.02475, 0.02484, 0.02445, 0.02491, 0.02465, 0.0246, 0.0245, 0.02432,
 0.02427, 0.02404, 0.02405, 0.02436, 0.02402, 0.02415, 0.02394, 0.02393, 0.02401, 0.02392,
 0.02409, 0.0241, 0.02379, 0.0241, 0.02381, 0.02383, 0.02355, 0.02353, 0.02369, 0.02355,
 0.02367, 0.02374, 0.02348, 0.02345, 0.02365, 0.0236, 0.0234, 0.02367, 0.02332, 0.0233,
 0.02325, 0.02321, 0.0232, 0.0232, 0.02331, 0.02295, 0.02312, 0.02293, 0.02327, 0.0228,
 0.02273, 0.02283, 0.02284, 0.02272, 0.02272, 0.02264, 0.02265, 0.0227, 0.02267, 0.02257,
 0.02253, 0.02255, 0.02243, 0.0229, 0.02242, 0.02235, 0.02248, 0.02248, 0.02204, 0.02249,
 0.02236, 0.02224, 0.0223, 0.02212, 0.02231, 0.02214, 0.02216, 0.02214, 0.02195, 0.02244,
 0.02223, 0.02201, 0.02204, 0.02206, 0.02198, 0.02193, 0.02195, 0.02211, 0.02203, 0.02188,
 0.02165, 0.02221, 0.0216, 0.02187, 0.02197, 0.0218, 0.02197, 0.02159, 0.02164, 0.02195,
 0.02166, 0.02167, 0.02171, 0.02162, 0.02151, 0.02162, 0.02122, 0.02194, 0.02129, 0.02126,
 0.02141, 0.02126, 0.02142, 0.0215, 0.02128, 0.02138, 0.02132, 0.02117, 0.0215, 0.0212,
 0.0216, 0.02128, 0.02125, 0.0212, 0.02119, 0.02123, 0.0213, 0.0212, 0.02125, 0.021,
 0.02083, 0.02119, 0.02098, 0.02126, 0.0209, 0.02097, 0.02117, 0.0209, 0.02112, 0.02067,
 0.02095, 0.02099, 0.02087, 0.02071, 0.02077, 0.02067, 0.02058, 0.02092, 0.02076, 0.02068,
 0.02028, 0.02083, 0.02071, 0.02076, 0.02078, 0.02077, 0.02058, 0.02022, 0.02056, 0.02087,
 0.02063, 0.02075, 0.02056, 0.02015, 0.02037, 0.02062, 0.02014, 0.02059, 0.0206, 0.02029,
 0.02007, 0.02057, 0.02026, 0.02023, 0.02034, 0.0201, 0.02033, 0.0202, 0.01997, 0.02013,
 0.02042, 0.02034, 0.0203, 0.0202, 0.02013, 0.02008, 0.02005, 0.02018, 0.01993, 0.02007,
 0.02005, 0.02018, 0.02009, 0.01995, 0.01965, 0.02004, 0.01982, 0.0199, 0.02011, 0.02004,
 0.01963, 0.02007, 0.01972, 0.01987, 0.01973, 0.01954, 0.01971, 0.01982, 0.01976, 0.02011,
 0.02016, 0.01959, 0.01983, 0.01994, 0.01983, 0.01989, 0.01967, 0.01974, 0.01978, 0.01973,
 0.01958, 0.01956, 0.01959, 0.01994, 0.01961, 0.01952, 0.01943, 0.0197, 0.01956, 0.01947,
 0.01949, 0.01926, 0.01971, 0.01955, 0.01961, 0.01943, 0.01923, 0.01979, 0.01937, 0.01954,
 0.01944, 0.01932, 0.0194, 0.01923, 0.01908, 0.01928, 0.01948, 0.01961, 0.01939, 0.01945,
 0.01934, 0.01935, 0.01902, 0.01959, 0.01928, 0.01896, 0.01952, 0.01913, 0.0194, 0.01921,
 0.0189, 0.01928, 0.0191, 0.01906, 0.01933, 0.01905, 0.01957, 0.01903, 0.01894, 0.01883
]

# ── SPORTS: Per-epoch validation metrics (101 checkpoints at eval_step=5) ──
sports_val_epochs = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90,
 95, 100, 105, 110, 115, 120, 125, 130, 135, 140, 145, 150, 155, 160, 165, 170,
 175, 180, 185, 190, 195, 200, 205, 210, 215, 220, 225, 230, 235, 240, 245, 250,
 255, 260, 265, 270, 275, 280, 285, 290, 295, 300, 305, 310, 315, 320, 325, 330,
 335, 340, 345, 350, 355, 360, 365, 370, 375, 380, 385, 390, 395, 400, 405, 410,
 415, 420, 425, 430, 435, 440, 445, 450, 455, 460, 465, 470, 475, 480, 485, 490, 495, 500]

sports_r10 = [0.0343, 0.0557, 0.0626, 0.065, 0.0671, 0.069, 0.0697, 0.0702, 0.0706, 0.071,
 0.0712, 0.0715, 0.0718, 0.072, 0.0723, 0.0727, 0.0726, 0.0727, 0.0726, 0.0728,
 0.0727, 0.0729, 0.0728, 0.0728, 0.0729, 0.0729, 0.073, 0.073, 0.0729, 0.073,
 0.073, 0.073, 0.0729, 0.073, 0.0729, 0.0729, 0.0729, 0.0729, 0.0729, 0.0729,
 0.0728, 0.0729, 0.0727, 0.0726, 0.0726, 0.0726, 0.0726, 0.0726, 0.0725, 0.0726,
 0.0726, 0.0726, 0.0727, 0.0726, 0.0726, 0.0726, 0.0726, 0.0726, 0.0726, 0.0727,
 0.0727, 0.0726, 0.0726, 0.0726, 0.0726, 0.0726, 0.0726, 0.0726, 0.0727, 0.0726,
 0.0726, 0.0726, 0.0727, 0.0726, 0.0726, 0.0726, 0.0726, 0.0727, 0.0727, 0.0727,
 0.0727, 0.0727, 0.0729, 0.0728, 0.0728, 0.0728, 0.0728, 0.0728, 0.0728, 0.0727,
 0.0727, 0.0727, 0.0727, 0.0727, 0.0727, 0.0727, 0.0727, 0.0727, 0.0727, 0.0727, 0.0727]

sports_r20 = [0.0511, 0.0839, 0.0944, 0.0983, 0.1007, 0.1032, 0.1041, 0.1049, 0.1054, 0.106,
 0.1062, 0.1065, 0.1067, 0.107, 0.1072, 0.1076, 0.1076, 0.1077, 0.1077, 0.1079,
 0.1079, 0.1079, 0.1079, 0.1079, 0.108, 0.108, 0.1082, 0.1082, 0.1081, 0.1082,
 0.1082, 0.1083, 0.1083, 0.1083, 0.1082, 0.1082, 0.1082, 0.1082, 0.1083, 0.1083,
 0.1082, 0.1082, 0.1082, 0.1081, 0.108, 0.108, 0.108, 0.108, 0.108, 0.1081,
 0.1081, 0.108, 0.1081, 0.108, 0.108, 0.108, 0.108, 0.1081, 0.108, 0.1082,
 0.1082, 0.1081, 0.1081, 0.1081, 0.1081, 0.1081, 0.1081, 0.1081, 0.1082, 0.108,
 0.108, 0.108, 0.1082, 0.108, 0.108, 0.108, 0.108, 0.1081, 0.1081, 0.1081,
 0.1082, 0.1082, 0.1083, 0.1082, 0.1083, 0.1082, 0.1082, 0.1082, 0.1082, 0.1082,
 0.1083, 0.1083, 0.1082, 0.1082, 0.1082, 0.1082, 0.1082, 0.1083, 0.1082, 0.1083, 0.1093]

sports_n10 = [0.018, 0.0301, 0.0342, 0.0357, 0.0368, 0.0379, 0.0382, 0.0386, 0.0389, 0.0392,
 0.0393, 0.0395, 0.0397, 0.0398, 0.0399, 0.0401, 0.0401, 0.0402, 0.0402, 0.0403,
 0.0403, 0.0403, 0.0403, 0.0404, 0.0404, 0.0404, 0.0405, 0.0405, 0.0405, 0.0406,
 0.0406, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406,
 0.0405, 0.0406, 0.0405, 0.0405, 0.0405, 0.0405, 0.0405, 0.0405, 0.0405, 0.0405,
 0.0405, 0.0405, 0.0406, 0.0405, 0.0405, 0.0405, 0.0405, 0.0406, 0.0405, 0.0406,
 0.0406, 0.0405, 0.0406, 0.0405, 0.0405, 0.0405, 0.0406, 0.0405, 0.0406, 0.0406,
 0.0405, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406, 0.0406,
 0.0406, 0.0406, 0.0408, 0.0407, 0.0408, 0.0407, 0.0408, 0.0407, 0.0407, 0.0407,
 0.0407, 0.0407, 0.0407, 0.0407, 0.0407, 0.0407, 0.0407, 0.0407, 0.0407, 0.0407, 0.0408]

sports_n20 = [0.0223, 0.0377, 0.0382, 0.0397, 0.0405, 0.0413, 0.0416, 0.042, 0.0423, 0.0428,
 0.0429, 0.0436, 0.0435, 0.0438, 0.044, 0.0442, 0.0443, 0.0444, 0.0447, 0.0449,
 0.0449, 0.045, 0.0453, 0.0455, 0.0456, 0.0458, 0.0459, 0.0459, 0.0459, 0.0459,
 0.0462, 0.0463, 0.0464, 0.0464, 0.0467, 0.047, 0.0469, 0.0469, 0.0469, 0.047,
 0.0472, 0.0471, 0.0469, 0.047, 0.047, 0.0469, 0.047, 0.0474, 0.0472, 0.0472,
 0.0473, 0.0472, 0.0472, 0.0472, 0.0474, 0.0475, 0.0475, 0.0476, 0.0474, 0.0474,
 0.0479, 0.0478, 0.0475, 0.0476, 0.0475, 0.0478, 0.0477, 0.0477, 0.0474, 0.0474,
 0.0473, 0.0476, 0.0477, 0.0476, 0.0479, 0.0477, 0.0478, 0.0479, 0.048, 0.0479,
 0.048, 0.0483, 0.048, 0.0482, 0.0479, 0.0481, 0.0479, 0.0478, 0.0477, 0.0477,
 0.0478, 0.048, 0.0481, 0.0479, 0.048, 0.048, 0.0484, 0.0483, 0.0483, 0.0481, 0.0482]

# ── SPORTS: Per-epoch training loss ────────────────────────────────────────
sports_loss_epochs = list(range(1, 501))
sports_loss_vals = [
 0.60904, 0.56604, 0.51249, 0.45677, 0.40519, 0.3614, 0.32461, 0.29442, 0.26912, 0.24786,
 0.23025, 0.21371, 0.20047, 0.18892, 0.17805, 0.16811, 0.16076, 0.15294, 0.14602, 0.14025,
 0.13322, 0.12871, 0.12322, 0.11823, 0.11444, 0.11023, 0.10729, 0.10339, 0.10055, 0.09815,
 0.09455, 0.09161, 0.08981, 0.0869, 0.08522, 0.08276, 0.081, 0.07937, 0.0775, 0.07598,
 0.07439, 0.07278, 0.07193, 0.06967, 0.06874, 0.06765, 0.06672, 0.06534, 0.06422, 0.06316,
 0.06199, 0.06174, 0.06, 0.05948, 0.05883, 0.05815, 0.05724, 0.0564, 0.05584, 0.05509,
 0.05438, 0.05373, 0.05312, 0.05237, 0.05205, 0.05123, 0.05118, 0.05075, 0.0497, 0.0493,
 0.04897, 0.04841, 0.04811, 0.04804, 0.04731, 0.04678, 0.04626, 0.04602, 0.04577, 0.04526,
 0.0449, 0.045, 0.04419, 0.04418, 0.04392, 0.0437, 0.04312, 0.04302, 0.04318, 0.0424,
 0.04263, 0.042, 0.04188, 0.04157, 0.04087, 0.04109, 0.04083, 0.04052, 0.04036, 0.04016,
 0.03996, 0.04014, 0.03971, 0.03944, 0.03915, 0.03924, 0.03913, 0.03897, 0.03874, 0.03853,
 0.03823, 0.03816, 0.03796, 0.03763, 0.03755, 0.03797, 0.03761, 0.03726, 0.03742, 0.03698,
 0.03685, 0.03677, 0.03656, 0.03653, 0.0368, 0.03634, 0.03639, 0.03592, 0.03601, 0.03582,
 0.03572, 0.03549, 0.03561, 0.03577, 0.03514, 0.03518, 0.03513, 0.0351, 0.03485, 0.03487,
 0.03459, 0.03486, 0.03488, 0.03419, 0.03425, 0.03388, 0.03409, 0.03396, 0.0339, 0.03362,
 0.03397, 0.03384, 0.03357, 0.03358, 0.03309, 0.03345, 0.03296, 0.03326, 0.03344, 0.03324,
 0.03311, 0.03283, 0.03295, 0.03258, 0.03283, 0.0327, 0.03284, 0.03269, 0.03264, 0.03259,
 0.03253, 0.0323, 0.03237, 0.03245, 0.03193, 0.03198, 0.03192, 0.03217, 0.03204, 0.03172,
 0.03182, 0.03174, 0.03156, 0.03163, 0.03183, 0.03172, 0.03124, 0.03141, 0.03136, 0.03156,
 0.0314, 0.03115, 0.03094, 0.03116, 0.03102, 0.03079, 0.03075, 0.03099, 0.03101, 0.03085,
 0.03068, 0.0308, 0.03062, 0.0309, 0.03047, 0.03048, 0.03065, 0.03059, 0.03048, 0.03053,
 0.03042, 0.03037, 0.03012, 0.03026, 0.03007, 0.03013, 0.03036, 0.03005, 0.03018, 0.02992,
 0.02994, 0.02952, 0.02988, 0.02999, 0.02992, 0.03002, 0.02947, 0.02972, 0.02936, 0.02938,
 0.02949, 0.02941, 0.02936, 0.02932, 0.02951, 0.02937, 0.02947, 0.02938, 0.02899, 0.02916,
 0.02925, 0.02912, 0.02899, 0.0289, 0.02909, 0.02893, 0.02891, 0.02893, 0.0293, 0.02915,
 0.02877, 0.0288, 0.02899, 0.02884, 0.02901, 0.02882, 0.02863, 0.02835, 0.02897, 0.02856,
 0.02861, 0.02857, 0.02858, 0.02865, 0.0283, 0.02879, 0.02886, 0.02855, 0.02835, 0.02838,
 0.02834, 0.02821, 0.02836, 0.02826, 0.02832, 0.0282, 0.02859, 0.02821, 0.02811, 0.02846,
 0.02812, 0.02845, 0.02833, 0.02822, 0.02819, 0.0282, 0.02793, 0.02805, 0.0283, 0.02829,
 0.02808, 0.02781, 0.02768, 0.02812, 0.02755, 0.02804, 0.02787, 0.02799, 0.02799, 0.02773,
 0.02769, 0.02782, 0.0278, 0.02768, 0.02776, 0.02771, 0.02757, 0.02745, 0.02767, 0.02782,
 0.02741, 0.02762, 0.02773, 0.02742, 0.02772, 0.02771, 0.0276, 0.02776, 0.02718, 0.02764,
 0.02723, 0.0274, 0.02741, 0.02759, 0.02719, 0.02733, 0.02736, 0.02716, 0.02729, 0.02736,
 0.02717, 0.0273, 0.02727, 0.0273, 0.02701, 0.02716, 0.02731, 0.02733, 0.02686, 0.02719,
 0.02702, 0.02709, 0.02693, 0.02752, 0.02703, 0.02705, 0.02682, 0.02702, 0.02693, 0.02685,
 0.02704, 0.02646, 0.02684, 0.02723, 0.02691, 0.02674, 0.02697, 0.0268, 0.02675, 0.02685,
 0.02704, 0.02694, 0.02705, 0.0267, 0.02683, 0.02667, 0.0268, 0.02684, 0.02654, 0.02647,
 0.02652, 0.02639, 0.02674, 0.0268, 0.02683, 0.02667, 0.02679, 0.0266, 0.0264, 0.02637,
 0.02645, 0.02662, 0.02648, 0.02668, 0.02671, 0.02651, 0.02636, 0.02637, 0.02654, 0.02644,
 0.02645, 0.02637, 0.02648, 0.02624, 0.0266, 0.02637, 0.02639, 0.02611, 0.02632, 0.02611,
 0.02647, 0.02652, 0.02636, 0.02616, 0.02641, 0.02621, 0.02628, 0.02612, 0.02639, 0.02621,
 0.02615, 0.0263, 0.02628, 0.02614, 0.02648, 0.02613, 0.02591, 0.0265, 0.02612, 0.02591,
 0.02585, 0.02615, 0.02611, 0.02597, 0.02616, 0.02629, 0.02624, 0.02604, 0.02598, 0.02606,
 0.02611, 0.02572, 0.02593, 0.02593, 0.02626, 0.0262, 0.02598, 0.02589, 0.02607, 0.02569,
 0.02598, 0.02571, 0.02575, 0.02601, 0.02592, 0.02609, 0.02569, 0.02564, 0.02577, 0.02591,
 0.02566, 0.02615, 0.02566, 0.02586, 0.02594, 0.02593, 0.02591, 0.02582, 0.02579, 0.02579,
 0.02584, 0.02574, 0.0255, 0.02571, 0.02588, 0.02585, 0.02541, 0.02549, 0.02542, 0.02559,
 0.0259, 0.02561, 0.02582, 0.02563, 0.02548, 0.02548, 0.02549, 0.02574, 0.02556, 0.02561,
 0.02557, 0.02538, 0.02568, 0.0258, 0.02558, 0.02577, 0.02539, 0.02554, 0.02561, 0.02565,
 0.02565, 0.02577, 0.02531, 0.02556, 0.02539, 0.02571, 0.02535, 0.02536, 0.02519, 0.02534
]

# ── BEST METRICS (hardcoded from log — single source of truth) ──────────────
best_baby_v3 = {
    'Recall@10': max(baby_r10),   # 0.0616 at epoch 260
    'Recall@20': max(baby_r20),   # 0.0945 at epoch 70
    'NDCG@10':   max(baby_n10),   # 0.0328 at epoch 260
    'NDCG@20':   max(baby_n20),   # 0.0411 at epoch 135 / 500
}
best_sports_v3 = {
    'Recall@10': max(sports_r10),  # 0.0730 at epoch 130
    'Recall@20': max(sports_r20),  # 0.1093 at epoch 500
    'NDCG@10':   max(sports_n10),  # 0.0408 at epoch 480
    'NDCG@20':   max(sports_n20),  # 0.0484 at epoch 495
}

print('=' * 70)
print('GROUND-TRUTH BEST METRICS (from hardcoded log arrays):')
print(f'  Baby  : {best_baby_v3}')
print(f'  Sports: {best_sports_v3}')
print('=' * 70)
print(f'Baby  val points: {len(baby_r20)}, loss points: {len(baby_loss_vals)}')
print(f'Sports val points: {len(sports_r20)}, loss points: {len(sports_loss_vals)}')


In [ ]:
# Cell 7: Learning Curves — actual per-epoch data from log
import numpy as np
import matplotlib.pyplot as plt

def smooth(values, w=10):
    if len(values) <= w: return values
    return np.convolve(values, np.ones(w)/w, mode='valid').tolist()

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('STAIR-v3 (ClipFuse) — Learning Curves (Ground-Truth Log Data)', fontsize=15, fontweight='bold')

for ci, (ds_name, loss_ep, loss_v, val_ep, r10, r20, n10, n20) in enumerate([
    ('Baby',   baby_loss_epochs,   baby_loss_vals,   baby_val_epochs,   baby_r10,   baby_r20,   baby_n10,   baby_n20),
    ('Sports', sports_loss_epochs, sports_loss_vals, sports_val_epochs, sports_r10, sports_r20, sports_n10, sports_n20),
]):
    # Trim to match lengths safely
    m_len = min(len(val_ep), len(r10), len(r20), len(n10), len(n20))
    v_ep, r10_c, r20_c, n10_c, n20_c = val_ep[:m_len], r10[:m_len], r20[:m_len], n10[:m_len], n20[:m_len]

    # Row 0: Loss
    ax = axes[0][ci]
    s = smooth(loss_v, w=10)
    offset = (10 - 1) // 2
    ax.plot(loss_ep, loss_v, color='#E8C55A', alpha=0.3, linewidth=0.8, label='Raw Loss')
    ax.plot(loss_ep[offset: offset + len(s)], s, color='#E8734A', linewidth=2, label='Smoothed (w=10)')
    ax.set_title(f'{ds_name} — BPR Training Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # Row 1: Recall
    ax = axes[1][ci]
    ax.plot(v_ep, r10_c, color='#2ECC71', linewidth=2.0, marker='o', markersize=2, label='Recall@10')
    ax.plot(v_ep, r20_c, color='#1A7A3F', linewidth=2.0, marker='s', markersize=2, label='Recall@20')
    ax.axhline(y=max(r10_c), color='#2ECC71', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axhline(y=max(r20_c), color='#1A7A3F', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.set_title(f'{ds_name} — Recall@K', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Recall'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.annotate(f'Best R@20={max(r20_c):.4f}', xy=(v_ep[r20_c.index(max(r20_c))], max(r20_c)),
                fontsize=7, color='#1A7A3F',
                xytext=(10, -15), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='#1A7A3F', lw=0.8))

    # Row 2: NDCG
    ax = axes[2][ci]
    ax.plot(v_ep, n10_c, color='#3498DB', linewidth=2.0, marker='o', markersize=2, label='NDCG@10')
    ax.plot(v_ep, n20_c, color='#1A5276', linewidth=2.0, marker='s', markersize=2, label='NDCG@20')
    ax.axhline(y=max(n10_c), color='#3498DB', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axhline(y=max(n20_c), color='#1A5276', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.set_title(f'{ds_name} — NDCG@K', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('NDCG'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.annotate(f'Best N@20={max(n20_c):.4f}', xy=(v_ep[n20_c.index(max(n20_c))], max(n20_c)),
                fontsize=7, color='#1A5276',
                xytext=(10, -15), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='#1A5276', lw=0.8))

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Learning curves saved successfully.')


In [ ]:
# Cell 8: Full Performance Comparison Table
from prettytable import PrettyTable

BASELINE = {
    'Baby':   {'Recall@10': 0.068560, 'Recall@20': 0.103420, 'NDCG@10': 0.036335, 'NDCG@20': 0.045143},
    'Sports': {'Recall@10': 0.074310, 'Recall@20': 0.111900, 'NDCG@10': 0.040200, 'NDCG@20': 0.050050},
}
V1_GCL = {
    'Baby':   {'Recall@10': 0.069459, 'Recall@20': 0.104700, 'NDCG@10': 0.036535, 'NDCG@20': 0.045531},
    'Sports': {'Recall@10': 0.074541, 'Recall@20': 0.112394, 'NDCG@10': 0.040429, 'NDCG@20': 0.050400},
}
V2_DYFUSE = {
    'Baby':   {'Recall@10': 0.057800, 'Recall@20': 0.089800, 'NDCG@10': 0.030900, 'NDCG@20': 0.038800},
    'Sports': {'Recall@10': 0.066900, 'Recall@20': 0.100200, 'NDCG@10': 0.036600, 'NDCG@20': 0.044900},
}
V3_CLIPFUSE = {
    'Baby':   best_baby_v3,
    'Sports': best_sports_v3,
}

METRICS  = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']
DATA_MAP = [BASELINE, V1_GCL, V2_DYFUSE, V3_CLIPFUSE]

print('=' * 115)
print('COMPARISON: Baseline  vs  v1 (GCL)  vs  v2 (DyFuse)  vs  v3 (ClipFuse)')
print('=' * 115)

for ds in ['Baby', 'Sports']:
    t = PrettyTable()
    t.field_names = ['Metric', 'Baseline', 'v1 (GCL)', 'v2 (DyFuse)', 'v3 (ClipFuse)', 'v3 vs Base', 'v3 vs v1', 'v3 vs v2']
    for metric in METRICS:
        vals = [data[ds].get(metric, 0) for data in DATA_MAP]
        v3 = vals[3]
        t.add_row([metric,
            f'{vals[0]:.6f}', f'{vals[1]:.6f}', f'{vals[2]:.6f}', f'{vals[3]:.6f}',
            f'{(v3-vals[0])/(vals[0]+1e-9)*100:+.2f}%',
            f'{(v3-vals[1])/(vals[1]+1e-9)*100:+.2f}%',
            f'{(v3-vals[2])/(vals[2]+1e-9)*100:+.2f}%',
        ])
    print(f'\nDataset: {ds}'); print(t)
print('=' * 115)


In [ ]:
# Cell 9: Bar Chart Comparison
import numpy as np
import matplotlib.pyplot as plt

METRICS  = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']
COLORS   = ['#95A5A6', '#3498DB', '#E74C3C', '#2ECC71']
LABELS   = ['Baseline', 'v1 (GCL)', 'v2 (DyFuse)', 'v3 (ClipFuse)']
DATA_MAP = [BASELINE, V1_GCL, V2_DYFUSE, V3_CLIPFUSE]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Model Comparison — Baseline vs v1 (GCL) vs v2 (DyFuse) vs v3 (ClipFuse)',
             fontsize=13, fontweight='bold')

for ri, ds in enumerate(['Baby', 'Sports']):
    for ci, metric in enumerate(METRICS):
        ax = axes[ri][ci]
        vals = [data[ds].get(metric, 0) for data in DATA_MAP]
        bars = ax.bar(LABELS, vals, color=COLORS, alpha=0.85, edgecolor='white', width=0.6)
        ax.set_title(f'{ds} — {metric}', fontweight='bold', fontsize=10)
        ax.set_ylim(0, max(vals) * 1.25)
        ax.tick_params(axis='x', rotation=25, labelsize=7)
        ax.grid(alpha=0.3, axis='y')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=8,
                    fontweight='bold' if val == max(vals) else 'normal')

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 10: Improvement Heatmap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

comparisons = [
    ('v1 vs Base', V1_GCL, BASELINE), ('v2 vs Base', V2_DYFUSE, BASELINE),
    ('v3 vs Base', V3_CLIPFUSE, BASELINE), ('v3 vs v1', V3_CLIPFUSE, V1_GCL),
    ('v3 vs v2', V3_CLIPFUSE, V2_DYFUSE),
]
rows = [f'{ds} — {m}' for ds in ['Baby', 'Sports'] for m in METRICS]
cols = [c[0] for c in comparisons]
data = np.zeros((len(rows), len(cols)))

for ci, (_, model, ref) in enumerate(comparisons):
    ri = 0
    for ds in ['Baby', 'Sports']:
        for metric in METRICS:
            m_val = model.get(ds, {}).get(metric, 0)
            r_val = ref.get(ds, {}).get(metric, 1e-9)
            data[ri, ci] = (m_val - r_val) / (r_val + 1e-9) * 100
            ri += 1

fig, ax = plt.subplots(figsize=(12, 8))
vmax = max(10, data.max() * 0.8)
im = ax.imshow(data, cmap='RdYlGn', aspect='auto',
               norm=mcolors.TwoSlopeNorm(vmin=data.min(), vcenter=0, vmax=vmax))
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, fontsize=10)
ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=9)
plt.colorbar(im, ax=ax, label='Relative Improvement (%)')
for i in range(len(rows)):
    for j in range(len(cols)):
        val = data[i, j]
        ax.text(j, i, f'{val:+.1f}%', ha='center', va='center', fontsize=9,
                fontweight='bold', color='white' if abs(val) > 8 else 'black')
ax.set_title('Relative Improvement (%) — STAIR-v3 vs Baselines', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/stair_v3_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 11: Final Summary
import os, shutil

print('=' * 70)
print('STAIR-v3 ClipFuse — Final Results (from hardcoded log arrays)')
print('=' * 70)

for ds, best in [('Baby', best_baby_v3), ('Sports', best_sports_v3)]:
    base = BASELINE[ds]
    print(f'\n[{ds}]')
    for metric in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
        v3 = best.get(metric, 0)
        bl = base[metric]
        d  = (v3 - bl) / (bl + 1e-9) * 100
        flag = '✓ BETTER' if v3 >= bl else '✗ WORSE'
        print(f'  {metric:<12}: v3={v3:.6f}  baseline={bl:.6f}  delta={d:+.2f}%  {flag}')

print('\n' + '=' * 70)
output_files = [
    '/kaggle/working/stair_v3_learning_curves.png',
    '/kaggle/working/stair_v3_comparison.png',
    '/kaggle/working/stair_v3_heatmap.png',
]
for fp in output_files:
    if os.path.exists(fp):
        print(f'  [OK]   {os.path.basename(fp):<50} {os.path.getsize(fp)/1024:.1f} KB')
    else:
        print(f'  [MISS] {os.path.basename(fp)}')
